# 📊 Dashboard UX para priorización de riesgos industriales · Piloto en Google Colab

**Notebook piloto (prueba de concepto)**

**Tema de tesis:** *Dashboard UX para priorización de riesgos industriales y toma de decisiones preventivas*
**Autores:** Reyes Salinas Alberto Renan · García Anzules Suanny Nayely
**Institución:** Universidad de Guayaquil · Facultad de Ingeniería Industrial · Sistemas de la Información

---

## ¿Qué demuestra este notebook?

La pregunta de investigación es: **¿qué visualizaciones facilitan la priorización de riesgos industriales por parte de usuarios técnicos y operativos?**
Este piloto recorre, de punta a punta, las tres partes del trabajo:

1. **Diseñar** las visualizaciones y armar un dashboard interactivo (plano de zonas críticas, matriz frecuencia × severidad, ranking, exposición y recomendaciones).
2. **Probar** la revisión heurística de Nielsen y el guion de tareas.
3. **Evaluar** con el experimento **A/B** (control sin dashboard vs. tratamiento con dashboard): tiempos, éxito, errores, precisión, SUS, pruebas estadísticas y tamaño del efecto.

### Relación con los objetivos específicos

| Objetivo específico | Pasos del notebook |
|---|---|
| **OE1.** Requisitos de usuarios y criterios de visualización | 1 y 2 |
| **OE2.** Diseñar e implementar un dashboard UX funcional | 3, 4 y 5 |
| **OE3.** Evaluar usabilidad y utilidad operativa y formular mejoras | 6, 7, 8 y 9 |

### Relación con la metodología (DSR + Human-Centred Design + Scrum)

| Elemento | Dónde se refleja |
|---|---|
| Historias de usuario HU-01…HU-08 | Paso 1 (cada una se vincula a un paso del notebook) |
| User-Centred Design (necesidades antes que pantallas) | Pasos 1 y 2 |
| Heurísticas de Nielsen | Paso 5 |
| Evaluación con usuarios (tareas, SUS, A/B) | Pasos 6 a 8 |
| Recomendaciones de rediseño basadas en evidencia | Paso 8 |

> ⚠️ **Aviso importante sobre los datos.** El dashboard consumirá datos de SI-02 (incidentes) y SI-01 (indicadores), que aún no están disponibles para este piloto, por lo que se **simulan**. Además, **los resultados de las “pruebas con usuarios” de los Pasos 6 a 8 también son simulados**: sirven para **validar y ensayar el procedimiento de análisis**, no son evidencia de la tesis. Las diferencias entre grupos se exageran a propósito para que se aprecie el método. Cuando tengan participantes reales, los Pasos 6 a 8 se reutilizan cambiando una sola variable (ver Paso 6).

### Cómo usarlo
**Entorno de ejecución → Ejecutar todo** (≈ 1 minuto). En el Paso 4 aparecen los controles interactivos del dashboard.

## Paso 0 · Preparación del entorno

| Librería | Uso |
|---|---|
| `plotly` | Visualizaciones interactivas del dashboard |
| `ipywidgets` | Filtros interactivos (zonas, tipo de evento, periodo, Top-N) |
| `scipy` | Pruebas estadísticas (Shapiro, Levene, t de Welch, Mann-Whitney, Fisher) |
| `pandas`, `numpy`, `matplotlib` | Datos, simulación y gráficos estadísticos |

In [ ]:
!pip install -q plotly ipywidgets scipy pandas numpy matplotlib

In [ ]:
import json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
pio.templates.default = "plotly_white"

# Necesario en Colab para que los widgets interactivos funcionen con gráficos
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

SEED = 2026
rng = np.random.default_rng(SEED)
print("Entorno listo ✔")

## Paso 0B · (Opcional) Conexión a Supabase: cargar y guardar datos

Esta sección permite **leer** los datos de entrada desde tu proyecto de Supabase (PostgreSQL) y **guardar** los resultados del notebook en él. Viene **desactivada** (`USAR_SUPABASE = False`) para que *Ejecutar todo* funcione sin credenciales.

**Cómo activarla**
1. En Colab, abre el panel 🔑 **Secrets** (izquierda) y crea `SUPABASE_URL` y `SUPABASE_KEY` (Supabase → *Project Settings → API*). Activa «Acceso desde el cuaderno».
2. Usa la clave **anon / public** con **Row Level Security (RLS)** activado. **Nunca** pegues la clave `service_role` en un notebook: da control total sobre la base y quien reciba el archivo podría usarla.
3. Crea las tablas (SQL de abajo, en el *SQL Editor* de Supabase) y cambia `USAR_SUPABASE = True`.

Se incluyen los dos métodos de tu celda original: **A** (solicitudes `requests`, el que usa el notebook) y **B** (SDK oficial). Además, la carga pagina de a 1 000 filas (el máximo por defecto de Supabase) y el guardado admite **upsert** para poder re-ejecutar sin duplicar filas.


### SQL para este notebook
```sql
-- Entradas del dashboard
create table if not exists accidentes_si02 (
  id text primary key, fecha date, zona text, tipo_evento text, severidad int, dias_perdidos int);
create table if not exists exposicion_si01 (
  zona text, mes text, exposicion_horas float8, primary key (zona, mes));

-- Resultados de la evaluación con usuarios (identificadores ANÓNIMOS: C01, T01…)
create table if not exists resultados_tareas (
  participante text, grupo text, perfil text, tarea text, tiempo_s float8, exito int, errores int, aciertos_zonas float8,
  primary key (participante, tarea));
create table if not exists resultados_sus (
  participante text primary key, s1 int, s2 int, s3 int, s4 int, s5 int, s6 int, s7 int, s8 int, s9 int, s10 int,
  utilidad int, claridad int, sus_puntaje float8);
create table if not exists resumen_participantes (
  participante text primary key, grupo text, perfil text, tiempo_total_s float8, tasa_exito float8,
  errores_total int, aciertos_t1 float8, precision_zonas float8);

-- Políticas de PRUEBA para un piloto (permisivas). Ajústalas antes de usar datos reales.
alter table accidentes_si02 enable row level security;
create policy "accidentes_si02_lectura"   on accidentes_si02 for select to anon using (true);
create policy "accidentes_si02_insercion" on accidentes_si02 for insert to anon with check (true);
create policy "accidentes_si02_update"    on accidentes_si02 for update to anon using (true) with check (true);

-- Políticas de PRUEBA para un piloto (permisivas). Ajústalas antes de usar datos reales.
alter table exposicion_si01 enable row level security;
create policy "exposicion_si01_lectura"   on exposicion_si01 for select to anon using (true);
create policy "exposicion_si01_insercion" on exposicion_si01 for insert to anon with check (true);
create policy "exposicion_si01_update"    on exposicion_si01 for update to anon using (true) with check (true);

-- Políticas de PRUEBA para un piloto (permisivas). Ajústalas antes de usar datos reales.
alter table resultados_tareas enable row level security;
create policy "resultados_tareas_lectura"   on resultados_tareas for select to anon using (true);
create policy "resultados_tareas_insercion" on resultados_tareas for insert to anon with check (true);
create policy "resultados_tareas_update"    on resultados_tareas for update to anon using (true) with check (true);

-- Políticas de PRUEBA para un piloto (permisivas). Ajústalas antes de usar datos reales.
alter table resultados_sus enable row level security;
create policy "resultados_sus_lectura"   on resultados_sus for select to anon using (true);
create policy "resultados_sus_insercion" on resultados_sus for insert to anon with check (true);
create policy "resultados_sus_update"    on resultados_sus for update to anon using (true) with check (true);

-- Políticas de PRUEBA para un piloto (permisivas). Ajústalas antes de usar datos reales.
alter table resumen_participantes enable row level security;
create policy "resumen_participantes_lectura"   on resumen_participantes for select to anon using (true);
create policy "resumen_participantes_insercion" on resumen_participantes for insert to anon with check (true);
create policy "resumen_participantes_update"    on resumen_participantes for update to anon using (true) with check (true);
```

In [ ]:
# Instalar el cliente oficial de Supabase (opcional: el MÉTODO A usa solo `requests`)
!pip install -q supabase

import os, json, requests
import pandas as pd

# ───────────── 1. Interruptor y credenciales ─────────────
USAR_SUPABASE = False     # ← cámbialo a True cuando ya tengas tus credenciales reales

def _secreto(nombre, defecto=""):
    """Busca la credencial en los Secrets de Colab (🔑 panel izquierdo) y, si no, en variables de entorno."""
    try:
        from google.colab import userdata
        valor = userdata.get(nombre)
    except Exception:
        valor = None
    return valor or os.environ.get(nombre, defecto)

# Project Settings → API (en tu panel de Supabase). Lo recomendable es guardarlas como Secrets.
SUPABASE_URL = _secreto("SUPABASE_URL", "https://xxxxxxxx.supabase.co")
SUPABASE_KEY = _secreto("SUPABASE_KEY", "")          # usa la clave «anon/public», NUNCA la «service_role»

def _supabase_listo():
    if not USAR_SUPABASE:
        print("ℹ️ Supabase desactivado (USAR_SUPABASE = False).")
        return False
    if "xxxxxxxx" in SUPABASE_URL or not SUPABASE_KEY:
        print("⚠️ Faltan credenciales: define SUPABASE_URL y SUPABASE_KEY (Secrets de Colab).")
        return False
    return True

def _headers(extra=None):
    h = {"apikey": SUPABASE_KEY, "Authorization": f"Bearer {SUPABASE_KEY}", "Content-Type": "application/json"}
    if extra:
        h.update(extra)
    return h

# ───────────── 2. CARGAR datos ─────────────
# MÉTODO A: solicitudes HTTP básicas (requests). Pagina de a 1000 filas (límite por defecto de Supabase).
def cargar_datos_api_rest(tabla, columnas="*", filtros=None, pagina=1000):
    """Equivale a SELECT <columnas> FROM <tabla>. `filtros` p. ej. {"zona": "eq.Z01"}."""
    if not _supabase_listo():
        return None
    print(f"📡 Consultando API REST de Supabase → {tabla} ...")
    filas, inicio = [], 0
    while True:
        r = requests.get(f"{SUPABASE_URL}/rest/v1/{tabla}", params={"select": columnas, **(filtros or {})},
                         headers=_headers({"Range-Unit": "items", "Range": f"{inicio}-{inicio + pagina - 1}"}), timeout=30)
        if r.status_code not in (200, 206):
            print(f"❌ Error en la llamada API: {r.status_code}\n{r.text}")
            return None
        lote = r.json()
        filas += lote
        if len(lote) < pagina:
            break
        inicio += pagina
    print(f"   {len(filas)} filas recibidas.")
    return pd.DataFrame(filas)

# MÉTODO B: cliente SDK oficial
def cargar_datos_sdk(tabla):
    if not _supabase_listo():
        return None
    print(f"📡 Conectando mediante SDK oficial → {tabla} ...")
    try:
        from supabase import create_client
        cliente = create_client(SUPABASE_URL, SUPABASE_KEY)
        filas, inicio, pagina = [], 0, 1000
        while True:
            lote = cliente.table(tabla).select("*").range(inicio, inicio + pagina - 1).execute().data
            filas += lote
            if len(lote) < pagina:
                break
            inicio += pagina
        return pd.DataFrame(filas)
    except Exception as e:
        print(f"❌ Error usando el SDK: {e}")
        return None

# ───────────── 3. GUARDAR datos ─────────────
def _a_registros(df):
    """DataFrame → lista de dicts lista para JSON (fechas ISO, NaN → null, enteros sin «.0»)."""
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_integer_dtype(df[c].dtype):            # incluye Int64 con nulos
            df[c] = pd.Series([None if pd.isna(v) else int(v) for v in df[c]], index=df.index, dtype=object)
        else:
            df[c] = df[c].map(lambda v: v.isoformat() if hasattr(v, "isoformat") and not pd.isna(v) else v)
    return json.loads(df.to_json(orient="records"))

def guardar_datos_api_rest(df, tabla, upsert_on=None, lote=500):
    """INSERT (o UPSERT si das `upsert_on`, p. ej. "id_evento" o "zona_codigo,periodo") por lotes."""
    if not _supabase_listo():
        return False
    registros = _a_registros(df)
    prefer = "return=minimal" + (",resolution=merge-duplicates" if upsert_on else "")
    for i in range(0, len(registros), lote):
        r = requests.post(f"{SUPABASE_URL}/rest/v1/{tabla}", params={"on_conflict": upsert_on} if upsert_on else None,
                          headers=_headers({"Prefer": prefer}), data=json.dumps(registros[i:i + lote]), timeout=60)
        if r.status_code not in (200, 201, 204):
            print(f"❌ Error guardando en «{tabla}» (lote {i // lote + 1}): {r.status_code}\n{r.text}")
            return False
    print(f"✅ {len(registros)} filas guardadas en «{tabla}»" + (f" (upsert por {upsert_on})." if upsert_on else "."))
    return True

def guardar_datos_sdk(df, tabla, upsert_on=None):
    if not _supabase_listo():
        return False
    try:
        from supabase import create_client
        cliente = create_client(SUPABASE_URL, SUPABASE_KEY)
        t = cliente.table(tabla)
        (t.upsert(_a_registros(df), on_conflict=upsert_on) if upsert_on else t.insert(_a_registros(df))).execute()
        print(f"✅ {len(df)} filas guardadas en «{tabla}» (SDK).")
        return True
    except Exception as e:
        print(f"❌ Error usando el SDK: {e}")
        return False

print("Funciones de Supabase listas ✔  ·  estado:", "ACTIVO" if USAR_SUPABASE else "desactivado (USAR_SUPABASE = False)")

## Paso 1 · Requisitos de usuario y guion de tareas (OE1)

Siguiendo **User-Centred Design**, primero se definen **quién** usa el dashboard y **qué decisiones** debe apoyar; las pantallas vienen después.

### Perfiles de usuario

| Perfil | Rol típico | Qué necesita decidir | Qué le estorba |
|---|---|---|---|
| **Técnico** | Analista / ingeniero de seguridad | Dónde se concentran las condiciones críticas y por qué (frecuencia, severidad, exposición) | Indicadores sin jerarquía; tener que consultar varias fuentes |
| **Operativo** | Supervisor / jefe de turno | Qué riesgo atender primero y qué acción preventiva aplicar | Tablas densas, términos poco claros, demasiados gráficos a la vez |

### Historias de usuario del anteproyecto y dónde se atienden en este piloto

In [ ]:
historias = pd.DataFrame([
    ["HU-01", "Técnico",   "Visualizar zonas críticas para identificar sectores de mayor riesgo",          "Alta",  1, "Paso 4 · Plano de zonas críticas"],
    ["HU-02", "Operativo", "Ver frecuencia y severidad para reconocer qué requiere atención prioritaria",  "Alta",  1, "Paso 4 · Matriz frecuencia × severidad y ranking"],
    ["HU-03", "Técnico",   "Consultar indicadores de exposición para interpretar las condiciones",         "Alta",  2, "Paso 4 · Gráfico exposición vs. riesgo"],
    ["HU-04", "Operativo", "Interfaz clara y fácil de usar, sin invertir mucho tiempo en interpretar",     "Alta",  2, "Paso 4 (diseño) y Paso 5 (heurísticas)"],
    ["HU-05", "Técnico",   "Recibir recomendaciones preventivas asociadas a los riesgos",                  "Media", 3, "Paso 4 · Tabla de recomendaciones"],
    ["HU-06", "Evaluador", "Interactuar con el dashboard mediante tareas definidas",                       "Alta",  3, "Paso 6 · Guion de tareas T1–T5"],
    ["HU-07", "Equipo",    "Medir tiempo, éxito, errores y satisfacción",                                  "Alta",  4, "Pasos 6 y 7"],
    ["HU-08", "Equipo",    "Generar recomendaciones de rediseño con base en los resultados",               "Media", 4, "Paso 8"],
], columns=["ID", "Rol", "Historia (resumen)", "Prioridad", "Iteración", "Dónde se atiende"])
historias

### Criterios de diseño de visualización (derivados de los perfiles)

1. **Jerarquía de lo general a lo específico:** indicadores clave → plano de zonas → riesgos priorizados → detalle y acciones.
2. **Un propósito por gráfico:** cada visualización responde *una* pregunta del guion de tareas.
3. **Color con significado y respaldo textual:** los cuatro niveles (Bajo, Medio, Alto, Crítico) usan colores de **luminosidad distinta** y **siempre** llevan su nombre escrito, de modo que no dependan solo del color (accesibilidad).
4. **Etiquetas directas** en lugar de leyendas lejanas; **solo se rotulan** los elementos prioritarios para evitar saturación.
5. **Acción al final:** toda lectura de riesgo termina en una recomendación preventiva.

## Paso 2 · Datos de entrada (simulados) y modelo de priorización

El dashboard **consume** datos; no los genera (alcance del anteproyecto). Se simulan con la forma que se espera de cada fuente:

| Fuente | Contenido simulado |
|---|---|
| **SI-02** (incidentes) | `id`, `fecha`, `zona`, `tipo_evento`, `severidad` (1-5), `dias_perdidos` |
| **SI-01** (indicadores) | `zona`, `mes`, `exposicion_horas` |
| Plano por zonas | Coordenadas rectangulares de cada zona dentro de la planta |
| Catálogo de riesgos | Recomendación preventiva por tipo de evento |

> 🔁 **Para usar datos reales:** cambia `USAR_DATOS_REALES = True` y apunta las rutas a los CSV de SI-02 y SI-01 (con las columnas indicadas arriba). El resto del notebook no cambia.

### Modelo de priorización del piloto

- **Riesgo = zona × tipo de evento.** Se clasifica con una matriz 5 × 5: **nivel de frecuencia** (eventos por mes) × **nivel de severidad** (promedio), ajustada ±20 % por la **exposición** de la zona.
- **Índice de zona (0-100):** severidad acumulada por cada 1 000 horas de exposición, relativo a la zona más cargada del periodo completo (así los colores no cambian de significado al filtrar).
- Niveles sobre el puntaje de riesgo: **Bajo** (≤ 4), **Medio** (≤ 9), **Alto** (≤ 15), **Crítico** (> 15). Sobre el índice de zona: < 25, < 50, < 75, ≥ 75.

> 📌 Esta lógica es **provisional**: el cálculo oficial de indicadores corresponde a **SI-01**. Aquí solo permite probar el diseño visual.

In [ ]:
USAR_DATOS_REALES = False
RUTA_SI02 = "si02_incidentes.csv"        # columnas: id, fecha, zona, tipo_evento, severidad, dias_perdidos
RUTA_SI01 = "si01_indicadores.csv"       # columnas: zona, mes, exposicion_horas

# ---------- Plano de la planta (coordenadas simuladas) ----------
ZONAS = pd.DataFrame([
    ["Z01", "Caldera",             2, 32, 28, 26],
    ["Z02", "Almacén de químicos", 32, 32, 30, 26],
    ["Z03", "Línea de ensamblaje", 64, 32, 34, 26],
    ["Z04", "Soldadura",            2,  2, 28, 26],
    ["Z05", "Bodega general",      32,  2, 30, 26],
    ["Z06", "Carga y descarga",    64,  2, 34, 26],
], columns=["zona", "nombre", "x0", "y0", "w", "h"])
NOMBRE_ZONA = dict(zip(ZONAS.zona, ZONAS.nombre))

TIPOS = ["Caída", "Golpe", "Quemadura", "Exposición química", "Atrapamiento", "Corte"]
SEV_ETIQUETA = {1: "Leve", 2: "Menor", 3: "Moderada", 4: "Grave", 5: "Crítica"}
MESES = [f"2026-{m:02d}" for m in range(1, 9)]

# Catálogo de riesgos: recomendaciones ORIENTATIVAS (deben validarse con el responsable de seguridad)
RECOMENDACIONES = {
    "Caída":              "Mantener pasillos despejados y señalizados; revisar barandas y antideslizantes.",
    "Golpe":              "Ordenar y estibar cargas; uso de casco; control del tráfico interno.",
    "Quemadura":          "Aislar superficies calientes; EPP térmico; permisos de trabajo en caliente.",
    "Exposición química": "Revisar hojas de seguridad (SDS); ventilación; duchas lavaojos; EPP químico.",
    "Atrapamiento":       "Resguardos en maquinaria; bloqueo y etiquetado (LOTO) antes de intervenir.",
    "Corte":              "Guantes anticorte; mantenimiento de herramientas; protecciones en filos.",
}

def cargar_o_simular():
    if USAR_DATOS_REALES:
        ev = pd.read_csv(RUTA_SI02, parse_dates=["fecha"])
        ind = pd.read_csv(RUTA_SI01)
        ev["mes"] = ev["fecha"].dt.strftime("%Y-%m")
        return ev, ind

    zonas = ZONAS.zona.tolist()
    p_zona = [0.24, 0.18, 0.10, 0.20, 0.12, 0.16]
    p_tipo = {   # qué tipo de evento predomina en cada zona
        "Z01": [0.10, 0.10, 0.45, 0.10, 0.20, 0.05], "Z02": [0.15, 0.05, 0.05, 0.65, 0.05, 0.05],
        "Z03": [0.10, 0.15, 0.05, 0.05, 0.35, 0.30], "Z04": [0.05, 0.10, 0.35, 0.10, 0.10, 0.30],
        "Z05": [0.40, 0.35, 0.05, 0.05, 0.05, 0.10], "Z06": [0.30, 0.40, 0.02, 0.03, 0.15, 0.10],
    }
    p_sev = {    # distribución de severidad 1-5 por zona (Z02 y Z01 son las más graves)
        "Z01": [0.25, 0.30, 0.25, 0.15, 0.05], "Z02": [0.10, 0.20, 0.30, 0.25, 0.15],
        "Z03": [0.40, 0.35, 0.15, 0.08, 0.02], "Z04": [0.20, 0.30, 0.28, 0.17, 0.05],
        "Z05": [0.45, 0.35, 0.13, 0.06, 0.01], "Z06": [0.30, 0.35, 0.20, 0.12, 0.03],
    }
    filas = []
    for i in range(1, 421):
        z = rng.choice(zonas, p=p_zona)
        sev = int(rng.choice([1, 2, 3, 4, 5], p=p_sev[z]))
        f = pd.Timestamp("2026-01-01") + pd.Timedelta(days=int(rng.integers(0, 243)))
        filas.append({"id": f"EV-{i:04d}", "fecha": f, "zona": z, "tipo_evento": rng.choice(TIPOS, p=p_tipo[z]),
                      "severidad": sev, "dias_perdidos": int([0, 1, 3, 8, 25][sev - 1] * rng.uniform(0.6, 1.4))})
    ev = pd.DataFrame(filas); ev["mes"] = ev["fecha"].dt.strftime("%Y-%m")

    horas_base = {"Z01": 4200, "Z02": 2600, "Z03": 5200, "Z04": 3000, "Z05": 2200, "Z06": 3600}
    ind = pd.DataFrame([{"zona": z, "mes": m, "exposicion_horas": float(int(rng.normal(horas_base[z], 250)))}
                        for z in zonas for m in MESES])
    return ev, ind

df_ev, df_ind = cargar_o_simular()
print(f"Incidentes (SI-02): {len(df_ev)} · Registros de exposición (SI-01): {len(df_ind)}")
df_ev.head()

### Opción: leer incidentes y exposición desde Supabase (en vez de los datos simulados)
Con `USAR_SUPABASE = True` el dashboard se alimenta de tus tablas. **Requisitos:** los códigos de zona de los datos deben existir en el plano (`ZONAS`, Paso 2); si no coinciden, se avisa y se conservan los datos simulados. Los tipos de evento y los meses se toman de los datos.

In [ ]:
TABLA_ACCIDENTES = "accidentes_si02"        # ← columnas: id, fecha, zona, tipo_evento, severidad, dias_perdidos  (p. ej. "nombreDetabla")
TABLA_EXPOSICION = "exposicion_si01"        # ← columnas: zona, mes, exposicion_horas

if USAR_SUPABASE:
    _ev, _ind = cargar_datos_api_rest(TABLA_ACCIDENTES), cargar_datos_api_rest(TABLA_EXPOSICION)
    cols_ev, cols_ind = ["id", "fecha", "zona", "tipo_evento", "severidad", "dias_perdidos"], ["zona", "mes", "exposicion_horas"]
    if (_ev is not None and _ind is not None and len(_ev) and len(_ind)
            and set(cols_ev) <= set(_ev.columns) and set(cols_ind) <= set(_ind.columns)):
        _ev["fecha"] = pd.to_datetime(_ev["fecha"]); _ev["mes"] = _ev["fecha"].dt.strftime("%Y-%m")
        _ev["dias_perdidos"] = _ev["dias_perdidos"].fillna(0)
        if set(_ev["zona"]) <= set(ZONAS["zona"]) and set(_ind["zona"]) <= set(ZONAS["zona"]):
            df_ev, df_ind = _ev[cols_ev + ["mes"]].copy(), _ind[cols_ind].copy()
            MESES = sorted(df_ind["mes"].unique())
            TIPOS = sorted(df_ev["tipo_evento"].unique())
            print(f"✅ Dashboard alimentado desde Supabase: {len(df_ev)} incidentes · {len(df_ind)} registros de exposición · {len(MESES)} meses")
        else:
            print("⚠️ Hay códigos de zona que no existen en el plano (ZONAS):",
                  sorted((set(_ev['zona']) | set(_ind['zona'])) - set(ZONAS['zona'])), "→ se mantienen los datos simulados.")
    else:
        print("⚠️ No se pudieron usar las tablas (vacías o con columnas distintas) → se mantienen los datos simulados.")
else:
    print("Se usan los datos SIMULADOS (USAR_SUPABASE = False).")

In [ ]:
# ---------- Lógica de priorización (provisional; el cálculo oficial es de SI-01) ----------
COLORES = {"Bajo": "#8fd19e", "Medio": "#ffe066", "Alto": "#f4a261", "Crítico": "#c1121f"}   # luminosidad decreciente
ORDEN_NIVEL = list(COLORES)

def nivel_puntaje(p):  return "Bajo" if p <= 4 else "Medio" if p <= 9 else "Alto" if p <= 15 else "Crítico"
def nivel_indice(i):   return "Bajo" if i < 25 else "Medio" if i < 50 else "Alto" if i < 75 else "Crítico"

# Factor de exposición 0.8–1.2 (referencia global, para que no cambie al filtrar)
_expo_media = df_ind.groupby("zona")["exposicion_horas"].mean()
FACTOR_EXPO = (0.8 + 0.4 * (_expo_media - _expo_media.min()) / (_expo_media.max() - _expo_media.min())).to_dict()

def _carga(ev, ind):
    sev = ev.groupby("zona")["severidad"].sum()
    horas = ind.groupby("zona")["exposicion_horas"].sum()
    return (sev.reindex(horas.index).fillna(0) / (horas / 1000))

CARGA_REF = _carga(df_ev, df_ind).max()          # referencia: zona más cargada del periodo completo

def calcular_zonas(ev, ind):
    z = ZONAS.copy()
    g = ev.groupby("zona").agg(n_eventos=("id", "size"), sev_media=("severidad", "mean")).reset_index()
    z = z.merge(g, on="zona", how="left").fillna({"n_eventos": 0, "sev_media": 0})
    horas = ind.groupby("zona")["exposicion_horas"].sum().rename("exposicion_total").reset_index()
    horas["exposicion_mensual"] = horas["exposicion_total"] / max(ind["mes"].nunique(), 1)
    z = z.merge(horas, on="zona", how="left")
    carga = _carga(ev, ind).rename("carga").reset_index()
    z = z.merge(carga, on="zona", how="left").fillna({"carga": 0})
    z["indice"] = (100 * z["carga"] / CARGA_REF).clip(upper=100)
    z["nivel"] = z["indice"].map(nivel_indice)
    return z

def calcular_riesgos(ev, ind):
    if ev.empty:
        return pd.DataFrame(columns=["zona", "tipo_evento", "n_eventos", "severidad_media", "frec_nivel", "sev_nivel",
                                     "puntaje", "nivel", "riesgo", "recomendacion"])
    n_meses = max(ind["mes"].nunique(), 1)
    r = ev.groupby(["zona", "tipo_evento"]).agg(n_eventos=("id", "size"), severidad_media=("severidad", "mean"),
                                                dias_perdidos=("dias_perdidos", "sum")).reset_index()
    por_mes = r["n_eventos"] / n_meses
    r["frec_nivel"] = pd.cut(por_mes, bins=[-np.inf, 0.5, 1, 2, 4, np.inf], labels=[1, 2, 3, 4, 5]).astype(int)
    r["sev_nivel"] = r["severidad_media"].round().clip(1, 5).astype(int)
    r["puntaje"] = (r["frec_nivel"] * r["sev_nivel"] * r["zona"].map(FACTOR_EXPO)).round(2)
    r["nivel"] = r["puntaje"].map(nivel_puntaje)
    r["riesgo"] = r["zona"].map(NOMBRE_ZONA) + " · " + r["tipo_evento"]
    r["recomendacion"] = r["tipo_evento"].map(RECOMENDACIONES).fillna("Definir la recomendación en el catálogo de riesgos.")
    return r.sort_values(["puntaje", "n_eventos"], ascending=False).reset_index(drop=True)

zonas_all = calcular_zonas(df_ev, df_ind)
riesgos_all = calcular_riesgos(df_ev, df_ind)
zonas_all[["zona", "nombre", "n_eventos", "sev_media", "exposicion_mensual", "indice", "nivel"]].round(1).sort_values("indice", ascending=False)

## Paso 3 · Biblioteca de visualizaciones (OE2)

Cada función genera **una** visualización y responde **una** pregunta del guion de tareas (que se define en el Paso 6):

| Visualización | Pregunta que responde | HU |
|---|---|---|
| **Indicadores clave** | ¿Cómo está la planta en general? | HU-04 |
| **Plano de zonas críticas** | ¿Dónde están los sectores más críticos? | HU-01 |
| **Matriz frecuencia × severidad** | ¿Qué eventos son a la vez frecuentes y graves? | HU-02 |
| **Ranking de riesgos (Top-N)** | ¿Cuál atiendo primero? | HU-02 |
| **Exposición vs. riesgo** | ¿La exposición explica el nivel de riesgo de la zona? | HU-03 |
| **Recomendaciones preventivas** | ¿Qué acción tomo? | HU-05 |

In [ ]:
def fig_kpis(ev, zonas, riesgos):
    top = riesgos.iloc[0] if len(riesgos) else None
    fig = make_subplots(rows=1, cols=4, specs=[[{"type": "indicator"}] * 4])
    fig.add_trace(go.Indicator(mode="number", value=len(ev), title={"text": "Eventos registrados"}), 1, 1)
    fig.add_trace(go.Indicator(mode="number", value=round(ev["severidad"].mean(), 2) if len(ev) else 0,
                               title={"text": "Severidad media (1-5)"}), 1, 2)
    fig.add_trace(go.Indicator(mode="number", value=int(zonas["nivel"].isin(["Alto", "Crítico"]).sum()),
                               number={"suffix": f" de {len(zonas)}"}, title={"text": "Zonas en nivel Alto o Crítico"}), 1, 3)
    fig.add_trace(go.Indicator(mode="number", value=float(top["puntaje"]) if top is not None else 0,
                               title={"text": "Riesgo prioritario<br><span style='font-size:0.75em'>"
                                              + (top["riesgo"] if top is not None else "—") + "</span>"}), 1, 4)
    fig.update_layout(height=170, margin=dict(l=10, r=10, t=40, b=0))
    return fig

def fig_plano(zonas, seleccion):
    fig = go.Figure()
    cx, cy, hover = [], [], []
    for r in zonas.itertuples():
        activa = r.zona in seleccion
        color = COLORES[r.nivel] if activa else "#e0e0e0"
        fig.add_shape(type="rect", x0=r.x0, y0=r.y0, x1=r.x0 + r.w, y1=r.y0 + r.h,
                      fillcolor=color, line=dict(color="#333333", width=2))
        txt = (f"<b>{r.nombre}</b><br>Índice {r.indice:.0f} · {r.nivel.upper()}" if activa
               else f"<b>{r.nombre}</b><br>(no seleccionada)")
        fig.add_annotation(x=r.x0 + r.w / 2, y=r.y0 + r.h / 2, text=txt, showarrow=False,
                           font=dict(size=13, color="white" if (activa and r.nivel == "Crítico") else "#111111"))
        cx.append(r.x0 + r.w / 2); cy.append(r.y0 + r.h / 2)
        hover.append(f"{r.nombre}<br>Eventos: {int(r.n_eventos)}<br>Severidad media: {r.sev_media:.2f}"
                     f"<br>Exposición: {0 if pd.isna(r.exposicion_mensual) else r.exposicion_mensual:,.0f} h/mes<br>Nivel: {r.nivel}")
    fig.add_trace(go.Scatter(x=cx, y=cy, mode="markers", marker=dict(size=60, opacity=0), hovertext=hover,
                             hoverinfo="text", showlegend=False))
    for nivel, c in COLORES.items():                                  # leyenda con nombre escrito
        fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name=nivel,
                                 marker=dict(size=14, color=c, symbol="square", line=dict(color="#333", width=1))))
    fig.update_xaxes(range=[0, 100], visible=False)
    fig.update_yaxes(range=[0, 60], visible=False, scaleanchor="x")
    fig.update_layout(title="Plano de zonas críticas", height=400, margin=dict(l=10, r=10, t=50, b=10),
                      legend=dict(orientation="h", y=-0.02, x=0.5, xanchor="center"))
    return fig

def fig_matriz(riesgos, top_n):
    fs = np.outer(np.arange(1, 6), np.ones(5)) * np.arange(1, 6)      # fila = severidad, columna = frecuencia
    fondo = {"Bajo": "#dff3e3", "Medio": "#fff4bf", "Alto": "#fbd8b8", "Crítico": "#efb3b8"}
    escala = [[0, fondo["Bajo"]], [0.146, fondo["Bajo"]], [0.146, fondo["Medio"]], [0.354, fondo["Medio"]],
              [0.354, fondo["Alto"]], [0.604, fondo["Alto"]], [0.604, fondo["Crítico"]], [1, fondo["Crítico"]]]
    fig = go.Figure(go.Heatmap(x=[1, 2, 3, 4, 5], y=[1, 2, 3, 4, 5], z=fs, zmin=1, zmax=25, colorscale=escala,
                               showscale=False, hoverinfo="skip", xgap=2, ygap=2))
    if len(riesgos):
        j = np.random.default_rng(0)
        x = riesgos["frec_nivel"] + j.uniform(-0.28, 0.28, len(riesgos))
        y = riesgos["sev_nivel"] + j.uniform(-0.28, 0.28, len(riesgos))
        etiqueta = [f"{z}·{t[:4]}" if i < top_n else "" for i, (z, t) in enumerate(zip(riesgos["zona"], riesgos["tipo_evento"]))]
        fig.add_trace(go.Scatter(
            x=x, y=y, mode="markers+text", text=etiqueta, textposition="top center", textfont=dict(size=10),
            marker=dict(size=10 + 3 * np.sqrt(riesgos["n_eventos"]), color=[COLORES[n] for n in riesgos["nivel"]],
                        line=dict(color="#222", width=1)),
            hovertext=[f"<b>{r}</b><br>Eventos: {n}<br>Puntaje: {p} ({nv})" for r, n, p, nv in
                       zip(riesgos["riesgo"], riesgos["n_eventos"], riesgos["puntaje"], riesgos["nivel"])],
            hoverinfo="text", showlegend=False))
    fig.update_xaxes(title="Nivel de frecuencia (1 baja → 5 alta)", tickvals=[1, 2, 3, 4, 5], range=[0.5, 5.5])
    fig.update_yaxes(title="Nivel de severidad (1 leve → 5 crítica)", tickvals=[1, 2, 3, 4, 5], range=[0.5, 5.5])
    fig.update_layout(title="Matriz frecuencia × severidad (tamaño = nº de eventos)", height=430, margin=dict(l=10, r=10, t=50, b=10))
    return fig

def fig_ranking(riesgos, top_n):
    t = riesgos.head(top_n).iloc[::-1]
    fig = go.Figure(go.Bar(
        x=t["puntaje"], y=t["riesgo"], orientation="h", text=[f"{p:.1f} · {n}" for p, n in zip(t["puntaje"], t["nivel"])],
        textposition="outside", marker=dict(color=[COLORES[n] for n in t["nivel"]], line=dict(color="#222", width=1)),
        hovertext=[f"Eventos: {n}<br>Severidad media: {s:.2f}" for n, s in zip(t["n_eventos"], t["severidad_media"])], hoverinfo="text"))
    for x0, lbl in [(4.5, "Medio"), (9.5, "Alto"), (15.5, "Crítico")]:
        fig.add_vline(x=x0, line_dash="dot", line_color="#888", annotation_text=lbl, annotation_position="top")
    fig.update_xaxes(title="Puntaje de riesgo", range=[0, max(30, (t["puntaje"].max() if len(t) else 0) * 1.25)])
    fig.update_layout(title=f"Riesgos priorizados (Top {len(t)})", height=max(260, 55 * len(t) + 110), margin=dict(l=10, r=30, t=50, b=10))
    return fig

def fig_exposicion(zonas, seleccion):
    z = zonas[zonas["zona"].isin(seleccion)]
    fig = go.Figure(go.Scatter(
        x=z["exposicion_mensual"], y=z["indice"], mode="markers+text", text=z["nombre"], textposition="top center",
        marker=dict(size=14 + 4 * np.sqrt(z["n_eventos"]), color=[COLORES[n] for n in z["nivel"]], line=dict(color="#222", width=1)),
        hovertext=[f"<b>{n}</b><br>Índice {i:.0f} ({nv})<br>Exposición {e:,.0f} h/mes" for n, i, nv, e in
                   zip(z["nombre"], z["indice"], z["nivel"], z["exposicion_mensual"])], hoverinfo="text"))
    fig.update_xaxes(title="Exposición (horas por mes)"); fig.update_yaxes(title="Índice de zona (0-100)", range=[0, 112])
    fig.update_layout(title="Exposición vs. riesgo (tamaño = nº de eventos)", height=380, margin=dict(l=10, r=10, t=50, b=10))
    return fig

def fig_recomendaciones(riesgos, top_n):
    t = riesgos.head(top_n)
    fig = go.Figure(go.Table(
        columnwidth=[0.5, 2.2, 1, 0.9, 5],
        header=dict(values=["#", "Riesgo", "Nivel", "Eventos", "Recomendación preventiva"], fill_color="#263238",
                    font=dict(color="white", size=13), align="left"),
        cells=dict(values=[list(range(1, len(t) + 1)), t["riesgo"], t["nivel"], t["n_eventos"], t["recomendacion"]],
                   fill_color=[["white"] * len(t), ["white"] * len(t), [COLORES[n] for n in t["nivel"]], ["white"] * len(t), ["white"] * len(t)],
                   font=dict(color=[["#111"] * len(t)] * 2 + [["white" if n == "Crítico" else "#111" for n in t["nivel"]]] + [["#111"] * len(t)] * 2, size=12),
                   align="left", height=30)))
    fig.update_layout(title="Acciones preventivas sugeridas", height=max(200, 38 * len(t) + 110), margin=dict(l=10, r=10, t=50, b=10))
    return fig

print("Funciones de visualización definidas ✔")

## Paso 4 · Dashboard interactivo (OE2)

El dashboard une las seis visualizaciones y añade **filtros**: zonas, tipos de evento, periodo y Top-N. Cada vez que se cambia un filtro, **todo se recalcula** (indicadores, niveles, ranking y recomendaciones).

> 💡 Pruébalo: desmarca zonas, acota el periodo o cambia el Top-N y observa cómo cambian el plano y el ranking.

In [ ]:
def filtrar(zonas_sel, tipos_sel, mes_ini, mes_fin):
    i0, i1 = MESES.index(mes_ini), MESES.index(mes_fin)
    meses_sel = MESES[min(i0, i1): max(i0, i1) + 1]
    ev = df_ev[df_ev.zona.isin(zonas_sel) & df_ev.tipo_evento.isin(tipos_sel) & df_ev.mes.isin(meses_sel)]
    ind = df_ind[df_ind.zona.isin(zonas_sel) & df_ind.mes.isin(meses_sel)]
    return ev, ind

def construir_figuras(zonas_sel, tipos_sel, mes_ini, mes_fin, top_n):
    ev, ind = filtrar(zonas_sel, tipos_sel, mes_ini, mes_fin)
    if ev.empty or ind.empty:
        return None
    zonas, riesgos = calcular_zonas(ev, ind), calcular_riesgos(ev, ind)
    # las zonas fuera de la selección se muestran en gris en el plano
    zonas_plano = calcular_zonas(ev, ind)
    return {
        "kpis": fig_kpis(ev, zonas[zonas.zona.isin(zonas_sel)], riesgos),
        "plano": fig_plano(zonas_plano, set(zonas_sel)),
        "matriz": fig_matriz(riesgos, top_n),
        "ranking": fig_ranking(riesgos, top_n),
        "exposicion": fig_exposicion(zonas, set(zonas_sel)),
        "recomendaciones": fig_recomendaciones(riesgos, top_n),
    }

def mostrar_dashboard(zonas_sel, tipos_sel, mes_ini, mes_fin, top_n):
    figs = construir_figuras(zonas_sel, tipos_sel, mes_ini, mes_fin, top_n)
    if figs is None:
        print("⚠️ Con esos filtros no hay datos. Amplía la selección de zonas, tipos o periodo.")
        return
    for f in figs.values():
        f.show()

# ----- Controles -----
w_zonas = widgets.SelectMultiple(options=[(f"{n} ({z})", z) for z, n in NOMBRE_ZONA.items()], value=tuple(NOMBRE_ZONA),
                                 description="Zonas", rows=6, layout=widgets.Layout(width="290px"))
w_tipos = widgets.SelectMultiple(options=TIPOS, value=tuple(TIPOS), description="Evento", rows=6, layout=widgets.Layout(width="290px"))
w_meses = widgets.SelectionRangeSlider(options=MESES, index=(0, len(MESES) - 1), description="Periodo",
                                       layout=widgets.Layout(width="420px"))
w_top = widgets.IntSlider(value=5, min=3, max=10, description="Top N", layout=widgets.Layout(width="420px"))
salida = widgets.Output()

def refrescar(*_):
    if not w_zonas.value or not w_tipos.value:
        with salida:
            clear_output(wait=True); print("Selecciona al menos una zona y un tipo de evento.")
        return
    with salida:
        clear_output(wait=True)
        mostrar_dashboard(list(w_zonas.value), list(w_tipos.value), w_meses.value[0], w_meses.value[1], w_top.value)

for w in (w_zonas, w_tipos, w_meses, w_top):
    w.observe(refrescar, names="value")

display(widgets.HBox([w_zonas, w_tipos, widgets.VBox([w_meses, w_top])]), salida)
refrescar()

### Exportar el dashboard como página web independiente

Para compartirlo con usuarios de prueba que **no** usan Colab, se genera un archivo `.html` (con los gráficos interactivos).

In [ ]:
def exportar_html(ruta="dashboard_riesgos_piloto.html", **filtros):
    f = dict(zonas_sel=list(NOMBRE_ZONA), tipos_sel=TIPOS, mes_ini=MESES[0], mes_fin=MESES[-1], top_n=5); f.update(filtros)
    figs = construir_figuras(**f)
    bloques = [fig.to_html(full_html=False, include_plotlyjs="cdn" if i == 0 else False) for i, fig in enumerate(figs.values())]
    pagina = f"""<!doctype html><html lang="es"><head><meta charset="utf-8">
<title>Dashboard de riesgos industriales (piloto)</title>
<style>body{{font-family:Arial,sans-serif;max-width:1100px;margin:24px auto;padding:0 16px;color:#222}}
h1{{font-size:1.5rem}} .nota{{background:#fff8e1;border-left:4px solid #f4a261;padding:8px 12px;margin:12px 0}}
section{{margin-bottom:8px}}</style></head><body>
<h1>Dashboard de riesgos industriales · versión piloto</h1>
<div class="nota">Datos simulados con fines de prueba. Periodo: {f['mes_ini']} a {f['mes_fin']}.</div>
{''.join(f'<section>{b}</section>' for b in bloques)}</body></html>"""
    Path(ruta).write_text(pagina, encoding="utf-8")
    return ruta

ruta_html = exportar_html()
print(f"Generado: {ruta_html}  ({Path(ruta_html).stat().st_size/1024:.0f} KB)")

## Paso 5 · Revisión heurística de Nielsen (antes de probar con usuarios)

El anteproyecto indica revisar la usabilidad con las **heurísticas de Nielsen** *antes* de las pruebas con usuarios, para corregir los problemas evidentes y no gastar participantes en ellos.

La columna **“Cómo se refleja en este piloto”** describe las decisiones de diseño reales de este dashboard. La columna **“Severidad (0-4)”** la completa el equipo tras revisar cada pantalla (escala de Nielsen: 0 = no es problema · 1 = cosmético · 2 = menor · 3 = mayor · 4 = catastrófico).

In [ ]:
heuristicas = pd.DataFrame([
    ["H1 · Visibilidad del estado del sistema",  "KPIs al inicio; el periodo y los filtros activos siempre visibles; el plano muestra en gris las zonas excluidas.",
     "¿El usuario sabe qué periodo y qué zonas está viendo?"],
    ["H2 · Correspondencia con el mundo real",   "Plano de planta con nombres reales de zonas; severidad con palabras (Leve…Crítica); niveles con nombre escrito.",
     "¿Los términos y la disposición coinciden con lo que conocen los operativos?"],
    ["H3 · Control y libertad del usuario",      "Filtros reversibles en cualquier momento; ningún filtro destruye información.",
     "¿Puede volver atrás sin esfuerzo?"],
    ["H4 · Consistencia y estándares",           "Una sola paleta de 4 niveles en todas las vistas; mismo orden Bajo → Crítico.",
     "¿El mismo color significa lo mismo en todos los gráficos?"],
    ["H5 · Prevención de errores",               "No se puede dejar el dashboard sin zonas/eventos: se avisa; el Top-N está acotado (3–10).",
     "¿Se evitan selecciones que producen pantallas vacías o engañosas?"],
    ["H6 · Diseño estético y minimalista",       "Un propósito por gráfico; solo se rotulan los Top-N en la matriz; sin elementos decorativos.",
     "¿Hay información que no ayuda a decidir?"],
], columns=["Heurística", "Cómo se refleja en este piloto", "Pregunta de verificación"])
heuristicas["Severidad (0-4)"] = pd.NA          # ← completar tras la revisión del equipo
heuristicas

In [ ]:
def resumen_heuristicas(df):
    sev = pd.to_numeric(df["Severidad (0-4)"], errors="coerce")
    if sev.isna().all():
        print("Aún no hay severidades registradas: completa la columna «Severidad (0-4)» y vuelve a ejecutar esta celda.")
        return
    print(f"Evaluadas: {sev.notna().sum()}/{len(df)} · Severidad media: {sev.mean():.2f} · "
          f"Problemas mayores (≥3): {', '.join(df.loc[sev >= 3, 'Heurística']) or 'ninguno'}")

# Ejemplo de uso (descomenta para completar):
# heuristicas.loc[0, "Severidad (0-4)"] = 1
resumen_heuristicas(heuristicas)

## Paso 6 · Diseño del experimento A/B y datos de la evaluación (OE3)

### Diseño
| Elemento | Definición |
|---|---|
| **Diseño** | Experimental A/B entre sujetos: cada persona realiza las **mismas cinco tareas** en una sola condición |
| **Grupo control** | Resuelve las tareas **sin** dashboard (con tablas planas de datos) |
| **Grupo tratamiento** | Resuelve las tareas **con** el dashboard |
| **Muestreo** | No probabilístico intencional (perfil técnico u operativo), repartido de forma equilibrada |
| **SUS y utilidad percibida** | Se aplican **solo al grupo tratamiento** (el SUS evalúa el sistema; el control no usa el dashboard) |

### Guion de tareas
Cada tarea se asocia a **una** visualización, de modo que el análisis pueda responder *qué visualizaciones facilitan la priorización*.

In [ ]:
TAREAS = pd.DataFrame([
    ["T1", "Identificar las 3 zonas más críticas de la planta",                       "Plano de zonas críticas",       "Precisión: nº de zonas correctas (0-3)"],
    ["T2", "Indicar qué tipo de evento es a la vez más frecuente y más grave",         "Matriz frecuencia × severidad", "Éxito / tiempo / errores"],
    ["T3", "Decir si la zona con más exposición es también la de mayor riesgo",        "Exposición vs. riesgo",         "Éxito / tiempo / errores"],
    ["T4", "Elegir el riesgo que requiere atención prioritaria",                       "Ranking de riesgos",            "Éxito / tiempo / errores"],
    ["T5", "Indicar la acción preventiva recomendada para ese riesgo",                 "Recomendaciones preventivas",   "Éxito / tiempo / errores"],
], columns=["Tarea", "Enunciado", "Visualización evaluada", "Métrica"])
TAREAS

In [ ]:
# Respuesta correcta de la T1 calculada a partir de los datos (¡el evaluador necesita la "clave"!)
TOP3_CORRECTO = zonas_all.sort_values("indice", ascending=False).head(3)["nombre"].tolist()
print("Clave de la tarea T1 (zonas más críticas):", TOP3_CORRECTO)
print("Clave de la tarea T4 (riesgo prioritario):", riesgos_all.iloc[0]["riesgo"])

### Resultados de la evaluación

La siguiente celda **simula** 12 personas por grupo para poder ensayar todo el análisis. Los parámetros están a la vista para que se entienda qué se está simulando.

> 🔁 **Para usar resultados reales:** pon `USAR_RESULTADOS_REALES = True`, completa el archivo `plantilla_resultados_tareas.csv` (se genera abajo con las columnas exactas) y súbelo. **No hay que tocar nada más**: los Pasos 7 y 8 se recalculan solos.
>
> 🔒 Con participantes reales: obtén **consentimiento informado** y trabaja con identificadores anónimos (C01, T01…), nunca con nombres.

In [ ]:
USAR_RESULTADOS_REALES = False
RUTA_TAREAS = "resultados_tareas.csv"       # columnas: participante, grupo, perfil, tarea, tiempo_s, exito, errores, aciertos_zonas
RUTA_SUS    = "resultados_sus.csv"          # columnas: participante, s1…s10, utilidad, claridad
N_POR_GRUPO = 12

# ---- Parámetros de la SIMULACIÓN (ilustrativos; el efecto se exagera a propósito) ----
SIM = {
    "tiempo_s":   {"control": [110, 125, 100, 115, 80], "tratamiento": [48, 70, 85, 42, 38]},      # media por tarea T1…T5
    "p_exito":    {"control": [None, 0.50, 0.55, 0.55, 0.60], "tratamiento": [None, 0.88, 0.72, 0.95, 0.92]},   # T1 usa aciertos
    "lam_error":  {"control": [1.2, 1.4, 1.0, 1.2, 0.8], "tratamiento": [0.3, 0.6, 0.7, 0.3, 0.3]},
    "p_zona":     {"control": 0.60, "tratamiento": 0.96},                                                # prob. de acertar cada zona en T1
}

def simular_resultados(n=N_POR_GRUPO, seed=SEED):
    r = np.random.default_rng(seed)
    filas, sus_filas = [], []
    for grupo, pref in (("control", "C"), ("tratamiento", "T")):
        for k in range(1, n + 1):
            pid = f"{pref}{k:02d}"
            perfil = "tecnico" if k % 2 else "operativo"
            habilidad = r.normal(0, 0.15)                              # unas personas son más rápidas que otras
            for j, t in enumerate(TAREAS["Tarea"]):
                tiempo = SIM["tiempo_s"][grupo][j] * np.exp(habilidad + r.normal(0, 0.22))
                errores = int(r.poisson(SIM["lam_error"][grupo][j]))
                if j == 0:
                    aciertos = int(r.binomial(3, SIM["p_zona"][grupo])); exito = int(aciertos == 3)
                else:
                    aciertos = np.nan; exito = int(r.random() < SIM["p_exito"][grupo][j])
                filas.append({"participante": pid, "grupo": grupo, "perfil": perfil, "tarea": t,
                              "tiempo_s": round(float(tiempo), 1), "exito": exito, "errores": errores, "aciertos_zonas": aciertos})
            if grupo == "tratamiento":                                  # SUS y percepciones: solo con dashboard
                s = r.normal(0, 0.5)
                fila = {"participante": pid}
                for i in range(1, 11):
                    base = 4.1 + s if i % 2 == 1 else 1.9 - s          # ítems impares positivos, pares negativos
                    fila[f"s{i}"] = int(np.clip(round(base + r.normal(0, 0.6)), 1, 5))
                fila["utilidad"] = int(np.clip(round(4.2 + 0.8 * s + r.normal(0, 0.6)), 1, 5))
                fila["claridad"] = int(np.clip(round(4.1 + 0.8 * s + r.normal(0, 0.6)), 1, 5))
                sus_filas.append(fila)
    return pd.DataFrame(filas), pd.DataFrame(sus_filas)

if USAR_RESULTADOS_REALES:
    df_tareas, df_sus_raw = pd.read_csv(RUTA_TAREAS), pd.read_csv(RUTA_SUS)
else:
    df_tareas, df_sus_raw = simular_resultados()

# Plantilla vacía con las columnas exactas para capturar datos reales
pd.DataFrame(columns=["participante", "grupo", "perfil", "tarea", "tiempo_s", "exito", "errores", "aciertos_zonas"]).to_csv("plantilla_resultados_tareas.csv", index=False)
pd.DataFrame(columns=["participante"] + [f"s{i}" for i in range(1, 11)] + ["utilidad", "claridad"]).to_csv("plantilla_resultados_sus.csv", index=False)

print(f"Registros de tareas: {len(df_tareas)} ({df_tareas.participante.nunique()} participantes) · Cuestionarios SUS: {len(df_sus_raw)}")
df_tareas.head(8)

## Paso 7 · Análisis estadístico (OE3)

### 7.1 Métricas por participante

| Métrica | Cómo se calcula | Mejor si… |
|---|---|---|
| Tiempo total | Suma del tiempo de las 5 tareas | es **menor** |
| Tasa de éxito | % de tareas completadas correctamente | es **mayor** |
| Errores totales | Suma de errores en las 5 tareas | es **menor** |
| Precisión de zonas críticas | Aciertos en T1 ÷ 3 | es **mayor** |

In [ ]:
df_part = (df_tareas.groupby(["participante", "grupo", "perfil"])
           .agg(tiempo_total_s=("tiempo_s", "sum"), tasa_exito=("exito", "mean"), errores_total=("errores", "sum"),
                aciertos_T1=("aciertos_zonas", "max"))
           .reset_index())
df_part["precision_zonas"] = df_part["aciertos_T1"] / 3

METRICAS = {   # nombre: (etiqueta, "menor" o "mayor" es mejor)
    "tiempo_total_s":  ("Tiempo total (s)", "menor"),
    "tasa_exito":      ("Tasa de éxito", "mayor"),
    "errores_total":   ("Errores totales", "menor"),
    "precision_zonas": ("Precisión de zonas críticas", "mayor"),
}
g = df_part.groupby("grupo")[list(METRICAS)].agg(["mean", "std", "median"]).round(2)
g

### 7.2 Supuestos y elección de la prueba

Se sigue la lógica del anteproyecto:

1. **Normalidad** (Shapiro-Wilk, por grupo) y **homogeneidad de varianzas** (Levene).
2. Si ambos grupos son aproximadamente normales → **t de Welch** (no exige varianzas iguales).
3. Si no (o la muestra es pequeña/discreta) → alternativa no paramétrica **U de Mann-Whitney**.
4. **Tamaño del efecto:** *d* de Cohen (y *g* de Hedges, corrección para muestras pequeñas) + **IC 95 % por bootstrap** de la diferencia de medias.
5. Como se hacen 4 contrastes, se reportan además los *p* **corregidos por Holm** (evita falsos positivos por comparaciones múltiples).

> 📝 **Nota metodológica.** El anteproyecto menciona la prueba de *Wilcoxon* como alternativa no paramétrica. Para **dos grupos independientes** (control vs. tratamiento) la prueba equivalente es **Mann-Whitney U** (también llamada *suma de rangos de Wilcoxon*). La prueba de *Wilcoxon de rangos con signo* es para datos **pareados** (mismas personas medidas dos veces). Conviene dejarlo así precisado en la tesis. Y si se evaluaran varias versiones del dashboard con las mismas personas, ahí sí aplicaría ANOVA de medidas repetidas o Wilcoxon pareado.

In [ ]:
def cohen_d(trat, ctrl):
    n1, n2 = len(trat), len(ctrl)
    sp = np.sqrt(((n1 - 1) * np.var(trat, ddof=1) + (n2 - 1) * np.var(ctrl, ddof=1)) / (n1 + n2 - 2))
    return (np.mean(trat) - np.mean(ctrl)) / sp if sp > 0 else np.nan

def hedges_g(d, n1, n2):
    return d * (1 - 3 / (4 * (n1 + n2) - 9))

def ic_bootstrap(trat, ctrl, n_boot=5000, seed=SEED):
    r = np.random.default_rng(seed)
    dif = [r.choice(trat, len(trat)).mean() - r.choice(ctrl, len(ctrl)).mean() for _ in range(n_boot)]
    return np.percentile(dif, [2.5, 97.5])

def holm(pvals):
    p = np.array(pvals, float); orden = np.argsort(p); ajust = np.empty_like(p); max_prev = 0
    for rank, idx in enumerate(orden):
        val = min(1, (len(p) - rank) * p[idx]); max_prev = max(max_prev, val); ajust[idx] = max_prev
    return ajust

def interpretar_d(d):
    a = abs(d)
    return "insignificante" if a < 0.2 else "pequeño" if a < 0.5 else "mediano" if a < 0.8 else "grande"

filas = []
for var, (etq, mejor) in METRICAS.items():
    c = df_part.loc[df_part.grupo == "control", var].to_numpy(float)
    t = df_part.loc[df_part.grupo == "tratamiento", var].to_numpy(float)
    sw_c, sw_t = stats.shapiro(c).pvalue, stats.shapiro(t).pvalue
    lev = stats.levene(c, t).pvalue
    normal = (sw_c > 0.05) and (sw_t > 0.05)
    if normal:
        prueba, res = "t de Welch", stats.ttest_ind(t, c, equal_var=False)
    else:
        prueba, res = "Mann-Whitney U", stats.mannwhitneyu(t, c, alternative="two-sided")
    d = cohen_d(t, c); g_ = hedges_g(d, len(t), len(c)); lo, hi = ic_bootstrap(t, c)
    filas.append({"métrica": etq, "mejor si es": mejor, "media control": c.mean(), "media tratamiento": t.mean(),
                  "Shapiro p (ctrl/trat)": f"{sw_c:.2f} / {sw_t:.2f}", "Levene p": lev, "prueba": prueba, "p": res.pvalue,
                  "d de Cohen": d, "g de Hedges": g_, "IC95% dif. medias": f"[{lo:.2f}; {hi:.2f}]", "efecto": interpretar_d(d)})

df_stats = pd.DataFrame(filas)
df_stats["p Holm"] = holm(df_stats["p"])
df_stats["¿mejora con dashboard?"] = [
    ("Sí" if ((r["media tratamiento"] < r["media control"]) if r["mejor si es"] == "menor" else (r["media tratamiento"] > r["media control"])) and r["p Holm"] < 0.05 else "No concluyente")
    for _, r in df_stats.iterrows()]
vista = df_stats.copy()
for col in ["media control", "media tratamiento", "Levene p", "d de Cohen", "g de Hedges"]:
    vista[col] = vista[col].round(2)
vista["p"] = vista["p"].map(lambda x: "< 0.001" if x < 0.001 else f"{x:.3f}")
vista["p Holm"] = vista["p Holm"].map(lambda x: "< 0.001" if x < 0.001 else f"{x:.3f}")
vista

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
colores_g = {"control": "#9e9e9e", "tratamiento": "#1f77b4"}
for ax, (var, (etq, mejor)) in zip(axes, METRICAS.items()):
    datos = [df_part.loc[df_part.grupo == g, var] for g in ("control", "tratamiento")]
    bp = ax.boxplot(datos, positions=[1, 2], widths=0.5, showfliers=False, patch_artist=True)
    for caja, g in zip(bp["boxes"], ("control", "tratamiento")):
        caja.set_facecolor(colores_g[g]); caja.set_alpha(0.35)
    for pos, g, d in zip((1, 2), ("control", "tratamiento"), datos):
        ax.scatter(np.random.default_rng(1).normal(pos, 0.06, len(d)), d, color=colores_g[g], s=28, zorder=3, edgecolor="white")
    ax.set_xticks([1, 2]); ax.set_xticklabels(["Control\n(sin dashboard)", "Tratamiento\n(con dashboard)"], fontsize=9)
    ax.set_title(f"{etq}\n(mejor si es {mejor})", fontsize=10)
plt.suptitle("Comparación A/B por participante (datos simulados)", y=1.03, fontsize=12, weight="bold")
plt.tight_layout(); plt.show()

### 7.3 Usabilidad percibida: SUS, utilidad y claridad (solo grupo tratamiento)

El **SUS** (*System Usability Scale*) son 10 afirmaciones de 1 a 5. Las **impares** son positivas (puntúan *respuesta − 1*) y las **pares** negativas (puntúan *5 − respuesta*); la suma se multiplica por 2,5 para obtener una nota de **0 a 100**. El valor de referencia internacional es **68**.

In [ ]:
def puntaje_sus(fila):
    imp = sum(fila[f"s{i}"] - 1 for i in (1, 3, 5, 7, 9))
    par = sum(5 - fila[f"s{i}"] for i in (2, 4, 6, 8, 10))
    return (imp + par) * 2.5

def calificacion_sus(x):         # aproximación de la escala de Sauro & Lewis
    return "A (excelente)" if x >= 80.8 else "B (bueno)" if x >= 74.1 else "C (aceptable)" if x >= 62.7 else "D (pobre)" if x >= 51.7 else "F (inaceptable)"

df_sus = df_sus_raw.copy()
df_sus["SUS"] = df_sus.apply(puntaje_sus, axis=1)

n = len(df_sus); media, sd = df_sus.SUS.mean(), df_sus.SUS.std(ddof=1)
ic = stats.t.interval(0.95, n - 1, loc=media, scale=sd / np.sqrt(n))
t_ref = stats.ttest_1samp(df_sus.SUS, 68, alternative="two-sided")

resumen_sus = pd.DataFrame([
    ["SUS medio (0-100)", f"{media:.1f}"],
    ["Desviación estándar", f"{sd:.1f}"],
    ["IC 95 % de la media", f"[{ic[0]:.1f}; {ic[1]:.1f}]"],
    ["Calificación aproximada", calificacion_sus(media)],
    ["Comparación con la referencia (68)", f"{'por encima' if media > 68 else 'por debajo'} · t de una muestra p = {t_ref.pvalue:.3f}"],
    ["Utilidad percibida (1-5)", f"{df_sus.utilidad.mean():.2f}"],
    ["Claridad de priorización (1-5)", f"{df_sus.claridad.mean():.2f}"],
], columns=["Indicador", "Resultado"])

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.hist(df_sus.SUS, bins=np.arange(0, 105, 5), color="#1f77b4", alpha=0.7, edgecolor="white")
ax.axvline(68, color="crimson", ls="--", label="Referencia 68"); ax.axvline(media, color="black", label=f"Media {media:.1f}")
ax.set_xlabel("Puntaje SUS"); ax.set_ylabel("participantes"); ax.set_xlim(0, 100); ax.legend(); ax.set_title("Distribución del SUS (simulado)")
plt.tight_layout(); plt.show()
resumen_sus

## Paso 8 · Pregunta de investigación: ¿qué visualizaciones facilitan la priorización? (OE3)

Como cada tarea evalúa **una** visualización, se puede comparar **tarea por tarea** el desempeño con y sin dashboard. Esto responde directamente la pregunta de la tesis y permite **triangular** con el SUS para formular **recomendaciones de rediseño** (HU-08).

**Reglas de decisión del piloto** (ajustables):
- Éxito del tratamiento **< 80 %** → *Rediseñar*.
- Reducción del tiempo **< 25 %** frente al control → *Revisar*.
- En otro caso → *Mantener*.

In [ ]:
UMBRAL_EXITO, UMBRAL_TIEMPO = 0.80, 0.25
viz_de_tarea = dict(zip(TAREAS["Tarea"], TAREAS["Visualización evaluada"]))

filas = []
for tarea, sub in df_tareas.groupby("tarea"):
    c, t = sub[sub.grupo == "control"], sub[sub.grupo == "tratamiento"]
    tabla = [[t.exito.sum(), len(t) - t.exito.sum()], [c.exito.sum(), len(c) - c.exito.sum()]]
    filas.append({
        "tarea": tarea, "visualización": viz_de_tarea[tarea],
        "éxito control": c.exito.mean(), "éxito tratamiento": t.exito.mean(),
        "p (Fisher)": stats.fisher_exact(tabla)[1],
        "tiempo control (s)": c.tiempo_s.mean(), "tiempo tratamiento (s)": t.tiempo_s.mean(),
        "reducción de tiempo": 1 - t.tiempo_s.mean() / c.tiempo_s.mean(),
        "errores tratamiento (media)": t.errores.mean(),
    })
df_viz = pd.DataFrame(filas)

REDISENO = {
    "Plano de zonas críticas":        "Resaltar el Top-3 con un borde y añadir el índice en grande dentro de cada zona.",
    "Matriz frecuencia × severidad":  "Reducir la superposición de burbujas, rotular solo el Top-N y explicar los ejes en lenguaje operativo.",
    "Exposición vs. riesgo":          "Marcar cuadrantes con una lectura esperada (p. ej. «alta exposición y alto riesgo») y simplificar el tamaño de burbuja.",
    "Ranking de riesgos":             "Mostrar los umbrales de nivel como líneas de referencia permanentes y destacar visualmente el primer lugar.",
    "Recomendaciones preventivas":    "Vincular cada recomendación con su fila del ranking y usar frases más breves y accionables.",
}

def decidir(fila):
    if fila["éxito tratamiento"] < UMBRAL_EXITO:        return "🔴 Rediseñar"
    if fila["reducción de tiempo"] < UMBRAL_TIEMPO:     return "🟠 Revisar"
    return "🟢 Mantener"

df_viz["decisión"] = df_viz.apply(decidir, axis=1)
df_viz["sugerencia de rediseño"] = [REDISENO[v] if d != "🟢 Mantener" else "—" for v, d in zip(df_viz["visualización"], df_viz["decisión"])]

vista = df_viz.copy()
for col in ["éxito control", "éxito tratamiento", "reducción de tiempo"]:
    vista[col] = (100 * vista[col]).round(0).astype(int).astype(str) + " %"
vista["p (Fisher)"] = vista["p (Fisher)"].round(3)
for col in ["tiempo control (s)", "tiempo tratamiento (s)", "errores tratamiento (media)"]:
    vista[col] = vista[col].round(1)
vista

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
x = np.arange(len(df_viz)); w = 0.38
axes[0].bar(x - w/2, 100 * df_viz["éxito control"], w, color="#9e9e9e", label="Control (sin dashboard)")
axes[0].bar(x + w/2, 100 * df_viz["éxito tratamiento"], w, color="#1f77b4", label="Tratamiento (con dashboard)")
axes[0].axhline(100 * UMBRAL_EXITO, color="crimson", ls="--", label=f"Umbral {int(100*UMBRAL_EXITO)} %")
axes[0].set_xticks(x); axes[0].set_xticklabels(df_viz["visualización"], rotation=18, ha="right", fontsize=9)
axes[0].set_ylabel("tareas exitosas (%)"); axes[0].set_ylim(0, 105); axes[0].legend(fontsize=8); axes[0].set_title("Éxito por visualización")

axes[1].bar(x - w/2, df_viz["tiempo control (s)"], w, color="#9e9e9e", label="Control")
axes[1].bar(x + w/2, df_viz["tiempo tratamiento (s)"], w, color="#1f77b4", label="Tratamiento")
axes[1].set_xticks(x); axes[1].set_xticklabels(df_viz["visualización"], rotation=18, ha="right", fontsize=9)
axes[1].set_ylabel("tiempo medio por tarea (s)"); axes[1].legend(fontsize=8); axes[1].set_title("Tiempo por visualización")
plt.tight_layout(); plt.show()

In [ ]:
# ---------- Triangulación: desempeño + usabilidad percibida → conclusiones preliminares ----------
mejor = df_viz.sort_values(["éxito tratamiento", "reducción de tiempo"], ascending=False).iloc[0]
peor  = df_viz.sort_values(["éxito tratamiento", "reducción de tiempo"], ascending=True).iloc[0]
mejoras = (df_stats["¿mejora con dashboard?"] == "Sí").sum()

print("TRIANGULACIÓN (resultados SIMULADOS — solo para ensayar el procedimiento)\n" + "-" * 78)
print(f"• Métricas globales con mejora significativa (Holm < 0.05): {mejoras} de {len(df_stats)}.")
print(f"• Visualización con mejor desempeño: {mejor['visualización']} (éxito {100*mejor['éxito tratamiento']:.0f} %, tiempo −{100*mejor['reducción de tiempo']:.0f} %).")
print(f"• Visualización con mayor oportunidad de mejora: {peor['visualización']} (éxito {100*peor['éxito tratamiento']:.0f} %).")
print(f"• Usabilidad global: SUS {media:.1f} → {calificacion_sus(media)}; utilidad percibida {df_sus.utilidad.mean():.1f}/5; claridad {df_sus.claridad.mean():.1f}/5.")
print("\nRecomendaciones de rediseño:")
rec = df_viz[df_viz["decisión"] != "🟢 Mantener"]
if rec.empty:
    print("  – Ninguna visualización incumple los umbrales; se mantiene el diseño actual.")
for _, r in rec.iterrows():
    print(f"  {r['decisión']} {r['visualización']}: {r['sugerencia de rediseño']}")

## Paso 9 · Entregables, limitaciones y próximos pasos

### Archivos generados

In [ ]:
df_part.to_csv("resumen_por_participante.csv", index=False)
df_stats.to_csv("resultados_pruebas_estadisticas.csv", index=False)
df_viz.to_csv("resultados_por_visualizacion.csv", index=False)
zonas_all.to_csv("zonas_indice.csv", index=False)
riesgos_all.to_csv("riesgos_priorizados.csv", index=False)

for f in ["dashboard_riesgos_piloto.html", "plantilla_resultados_tareas.csv", "plantilla_resultados_sus.csv",
          "resumen_por_participante.csv", "resultados_pruebas_estadisticas.csv", "resultados_por_visualizacion.csv",
          "zonas_indice.csv", "riesgos_priorizados.csv"]:
    print(f"  • {f:42s} {Path(f).stat().st_size/1024:7.1f} KB")

# En Colab: descarga directa, p. ej.
# from google.colab import files; files.download("dashboard_riesgos_piloto.html")

### Guardar los resultados de la evaluación en Supabase
Guarda los resultados por tarea, los cuestionarios SUS y el resumen por participante. Para evitar mezclar **datos simulados** con datos reales de la tesis, **por defecto no se suben resultados simulados**: solo se suben si `USAR_RESULTADOS_REALES = True` (Paso 6) o si activas `SUBIR_SIMULADOS` a propósito. Recuerda usar **identificadores anónimos** y contar con **consentimiento informado**.

In [ ]:
SUBIR_SIMULADOS = False          # True solo para probar el flujo con los resultados simulados

if USAR_SUPABASE and (USAR_RESULTADOS_REALES or SUBIR_SIMULADOS):
    sus_out = df_sus.rename(columns={"SUS": "sus_puntaje"})
    part_out = df_part.rename(columns={"aciertos_T1": "aciertos_t1"})
    guardar_datos_api_rest(df_tareas, "resultados_tareas", upsert_on="participante,tarea")
    guardar_datos_api_rest(sus_out, "resultados_sus", upsert_on="participante")
    guardar_datos_api_rest(part_out, "resumen_participantes", upsert_on="participante")
elif USAR_SUPABASE:
    print("ℹ️ Los resultados son SIMULADOS: no se subieron. (Activa SUBIR_SIMULADOS = True solo para pruebas.)")
else:
    print("Supabase desactivado: nada se envió. (Cambia USAR_SUPABASE = True para guardar.)")

### Qué quedó demostrado
- El dashboard puede construirse como **capa de interacción** sobre datos de SI-01/SI-02, con filtros y una jerarquía visual clara, sin necesidad de un gemelo digital completo.
- La **cadena de evaluación completa** (tareas → métricas → supuestos → prueba → tamaño del efecto → corrección múltiple → SUS → decisiones de rediseño) funciona y se recalcula automáticamente con solo cambiar el origen de los datos.
- Cada tarea está atada a **una visualización**, lo que permite responder la pregunta de la tesis con evidencia y no solo con la opinión global.

### Limitaciones que conviene declarar
- **Todos los datos son simulados** (eventos, exposición y resultados de usuarios). Nada de lo calculado en los Pasos 7 y 8 es un hallazgo de la tesis.
- La **lógica de priorización es provisional**; debe sustituirse por los indicadores oficiales de SI-01 y validarse con un responsable de seguridad (igual que las recomendaciones preventivas).
- Con **12 participantes por grupo** los p-valores son indicativos: reportar siempre el tamaño del efecto y su intervalo de confianza, e interpretar sin generalizar (como indica el anteproyecto).
- El SUS está pensado para evaluar **un sistema**; por eso solo lo responde el grupo tratamiento. Si desean comparar percepción entre grupos, hace falta un instrumento común (p. ej. NASA-TLX o un cuestionario propio).
- Falta el **UEQ**, mencionado como alternativa en el anteproyecto (tiene versión en español).

### Próximos pasos sugeridos
1. Conectar los CSV reales de SI-02 y SI-01 (`USAR_DATOS_REALES = True`) o, mejor aún, consumir la API de IND-SIS-02/SIS-API-01.
2. Completar la revisión heurística del Paso 5 y corregir lo que salga con severidad ≥ 3 **antes** de reclutar participantes.
3. Reclutar usuarios técnicos y operativos, aplicar el guion T1–T5 y capturar resultados en las plantillas CSV.
4. Definir con anticipación el **criterio de éxito** (p. ej. SUS ≥ 68, éxito ≥ 80 %) para no ajustar los umbrales después de ver los resultados.
5. Pasar de este piloto a un dashboard desplegable (Plotly Dash o Streamlit) reutilizando las funciones del Paso 3.